In [14]:
import json
import os
import pandas as pd
import time
import requests
os.chdir("/Users/withers/GitProjects/OTAR3088")  # adjusting to allow imports from /Data_mining

from bs4 import BeautifulSoup as bs
from pprint import pprint
from typing import Optional
from Data_mining.labelstudio_e2e.labelstudio_parse import clean_text, filter_tags, get_epmc_full_text, get_ppr_abstract, peek_at_xml, process_sections
# !!! Reminder - in order to try out conversion of ids to PMCIDs, the env vars 'NCBI_API_KEY & 'NCBI_EMAIL' would ideally be set

### Notes

- [ ] Push to OTAR3088 branch, import repo and merge from there to personal

- [x] List OTs PMCIDs

- [ ] Rename labelstudio_parse -> ePMC utils & relocate LS code

- [x] Grab valid texts, keep year range & journal list

    - [x] Yes for ppr, not for full texts

- [x] Text -> cleaned & senticized

    - [x] Not for ppr, yes for full texts

- [x] ePMC annotations loaded (Open Targets annotation as source)

- [ ] Models - cellate V2 (for now) & disease

- [ ] Coocurrences / relx

- [ ] With & without grounding

- [ ] Visualise 


Aim - Show the use of our data

- Keep with OTs papers 
- Pulling the ePMC annotations, also using cellate

Next up for data:
- ChEBI

- - - - - - - - - - -
E.g. (Cell)-[literature_evidence]-(Disease)
     - Name   - Freq               - Name
     - Tissue - SVO                - Family


Metadata:
- "What cell types are associated with x disease?"
--------
- Frequency of pair occurrences by sentence (weak association)
    - How many papers report 
- Relx would be subject - verb - object (stronger association)
--------
- LLM inferred relationships: Can we scale to this?

### Loading / parsing data

In [ ]:
# Open targets article IDs, 2023-25
# Sourced from https://www.opentargets.org/publications
uids = {
    "38165445": "PMID",
    "40316700": "PMID",
    "PMC11701534": "PMCID",
    "PMC11387188": "PMCID",
    "PMC11264729": "PMCID",
    "PMC11116453": "PMCID",
    "PMC11153150": "PMCID",
    "PMC11223857": "PMCID",
    "PMC11023940": "PMCID",
    "PMC10789043": "PMCID",
    "38215750": "PMID",
    "38717300": "PMID",
    "PMC10520009": "PMCID",
    "PMC10538656": "PMCID",
    "PPR721544": "PPR",
    "PPR1068204": "PPR",
    "PPR1041141": "PPR",
    "PPR968092": "PPR",
    "PMC12268933": "PMCID",
    "PMC12436637": "PMCID",
    "PMC10350819": "PMCID",
    "PMC10287956": "PMCID",
    "PMC9998435": "PMCID",
    "PMC10011132": "PMCID",
    "PMC9942875": "PMCID",
    "PMC9825572": "PMCID",
    "PMC12483672": "PMCID",
    "PMC12491431": "PMCID",
    "PMC11968318": "PMCID"
    }

In [ ]:
def grab_epmc_metadata(identifier: str, id_type: str):
    """
    Query Europe PMC for metadata using a described identifier
    Identifiers can be PMCID, PMID or PPR ids
    e.g. identifier = "PMC7614751", id_type="PMCID".
    """
    base_url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    if id_type == "PMCID":
        query = identifier
    else:
        query = f'EXT_ID:"{identifier}"'
    params = {
        "query": query,
        "resultType": "core",
        "pageSize": 1,
        "format": "json",
    }
    resp = requests.get(base_url, params=params)
    resp.raise_for_status()
    data = resp.json()

    if data.get("hitCount", 0) == 0:
        print(f"No result found for {id_type} '{identifier}'")
        return None

    res = data["resultList"]["result"][0]
    res_dict = {
            "doi": res.get("doi"),
            "title": res.get("title"),
            "year": res.get("pubYear"),
            "authors": res.get("authorString"),
            "source": res.get("source"),
        }
    res_dict["identifier"] = identifier
    res_dict["identifier_type"] = id_type
    try:
        res_dict["journal"] = res.get("journalInfo")["journal"]["title"]
    except:
        res_dict["journal"] = None # Will be None for PPR articles

    res_dict["abstract"] = clean_text(res.get("abstractText")),
    return res_dict

article_texts = []
full_texts = []
for uid in uids.keys():
    uid_metadata = grab_epmc_metadata(identifier=uid, id_type=uids[uid])
    if uids[uid] == "PMCID":
        # Collect full text and associated metadata
        fulltext_xml = get_epmc_full_text(uid, uids)
        if fulltext_xml != None:
            soup = bs(fulltext_xml, 'lxml')
            fulltext_clean = filter_tags(soup)
            # TODO - Don't need section processing in this form for this work - edit
            fulltext_clean_list = process_sections(fulltext_clean, uid) # () Detailing each section header + paragraph through article
            # peek_at_xml(fulltext_clean) # Useful if returned text is acting up
            full_texts.append(fulltext_clean_list)
    else:
        full_texts.append(None) # Full text not being accessed for non-PMCID articles
    article_texts.append(uid_metadata)

article_texts_df = pd.DataFrame(data=article_texts)
article_texts_df["full_text"] = full_texts
        

### Data stats & visualisation

In [86]:
# Quick data stats

print(f"{len(article_texts_df)} Open Targets articles have been pulled for testing, having been manually sourced from https://www.opentargets.org/publications.")
print(f"""\t- Selected articles were published (or written) between {min(article_texts_df["year"])} and {max(article_texts_df["year"])}
      \t- Published across {len(set(article_texts_df["journal"]))} journals, with the exception of {len(article_texts_df[article_texts_df["identifier_type"] == "PPR"])} that are pre-published.
      \t- Full texts were accessible for {len(article_texts_df[article_texts_df["full_text"].notna()])} PMC articles, with the rest having abstracts collected""")

29 Open Targets articles have been pulled for testing, having been manually sourced from https://www.opentargets.org/publications.
	- Selected articles were published (or written) between 2023 and 2025
      	- Published across 18 journals, with the exception of 4 that are pre-published.
      	- Full texts were accessible for 21 PMC articles, with the rest having abstracts collected


In [78]:
article_texts_df.sort_values(by="year", ascending=False)

,doi,title,year,authors,source,identifier,identifier_type,journal,abstract,full_text
28,10.1093/bioinformatics/btaf070,"Associations on the Fly, a new feature aiming ...",2025,"Cruz-Castillo C, Fumis L, Mehta C, Martinez-Os...",MED,PMC11968318,PMCID,"Bioinformatics (Oxford, England)",(<h4>Motivation</h4>The Open Targets Platform ...,"[(PMC11968318, Associations on the Fly, a new ..."
2,10.1093/nar/gkae1128,Open Targets Platform: facilitating therapeuti...,2025,"Buniello A, Suveges D, Cruz-Castillo C, Llinar...",MED,PMC11701534,PMCID,Nucleic acids research,(The Open Targets Platform (https://platform.o...,"[(PMC11701534, Open Targets Platform: facilita..."
27,10.1038/s41467-025-63856-7,Trans-eQTL mapping prioritises USP18 as a nega...,2025,"Freimann K, Brümmer A, Warmerdam R, Rupall TS,...",MED,PMC12491431,PMCID,Nature communications,(Although genome-wide association studies have...,"[(PMC12491431, Trans-eQTL mapping prioritises ..."
26,10.1016/j.xpro.2025.104111,Protocol for pooled FACS-based CRISPR knockout...,2025,"Washer SJ, Navarro-Guerrero E, Cowley SA, Ebne...",MED,PMC12483672,PMCID,STAR protocols,"(Here, we present a protocol for CRISPR knocko...","[(PMC12483672, Protocol for pooled FACS-based ..."
19,10.1038/s44318-025-00514-0,Tau uptake by human neurons depends on recepto...,2025,"Evans LD, Strano A, Tuck E, Campbell A, Smith ...",MED,PMC12436637,PMCID,The EMBO journal,(Extracellular release and uptake of pathogeni...,"[(PMC12436637, Tau uptake by human neurons dep..."
18,10.1016/j.isci.2025.112897,Simulating CD8 T cell exhaustion: A comprehens...,2025,"Manrique-Rincón AJ, Foster B, Horswell S, Goul...",MED,PMC12268933,PMCID,iScience,(Immunotherapy has transformed cancer treatmen...,"[(PMC12268933, Simulating CD8 T cell exhaustio..."
17,10.1101/2025.01.14.632908,Decoding MASLD Progression: A Molecular Trajec...,2025,"Kamzolas I, Koutsandreas T, Barker CG, Vathrak...",PPR,PPR968092,PPR,None,(Metabolic Dysfunction-Associated Steatotic Li...,None
16,10.1101/2025.06.24.25330216,Cell-type-resolved genetic regulatory variatio...,2025,"Alegbe T, Harris BT, Fachal L, Ramirez-Navarro...",PPR,PPR1041141,PPR,None,(Most genetic variants associated with complex...,None
15,10.1101/2025.08.18.670767,Integrated QTL mapping and CRISPR screening in...,2025,"Perez-Alcantara M, Washer S, Chen Y, Steer J, ...",PPR,PPR1068204,PPR,None,(Mounting evidence implicates microglia in neu...,None
1,10.1038/s41587-025-02659-z,A tissue-specific atlas of protein-protein ass...,2025,"Laman Trip DS, van Oostrum M, Memon D, Frommel...",MED,40316700,PMID,Nature biotechnology,(Despite progress in mapping protein-protein i...,None


In [119]:
from IPython.display import display, HTML, JSON
article_text = article_texts_df.iloc[3]

def display_text_nicely(title, text, max_chars=1000):
    preview = text[:max_chars] + "..." if len(text) > max_chars else text
    display(HTML(f"""
    <details>
    <summary><strong>{title}</strong></summary>
    <div style="padding: 15px; background: #f8f9fa; 
                border-left: 4px solid #007acc; margin: 10px 0;">
        <p style="font-size: 14px; line-height: 1.6; white-space: pre-wrap;">
            {preview}
        </p>
        {'<br><small><em>Click to see full text</em></small>' if len(text) > max_chars else ''}
    </div>
    </details>
    """))

# List out to view abstracts only
# Click to expand
for row in article_texts_df.itertuples():
    display_text_nicely(f"ABSTRACT '{row.identifier}' | {row.title}", row.abstract)

### Gathering existing OTs annotations

In [17]:
resp = requests.get("https://www.ebi.ac.uk/europepmc/annotations_api/annotationsByArticleIds?articleIds=PMC%3APMC11116453&provider=OpenTargets&format=JSON")
# print(resp._content)
data_str = resp.content.decode("utf-8")    # convert bytes → text
data_json = json.loads(data_str)

- [x] Decode response
- [x] Response = [] of articles
- [x] Item in list -> 'exact' for text, 'tags' -> List of annotations
- [x] Pass if text in seen []
- [ ] Annotate 'exact' text with cellate
- [x] Add text to seen []
- [ ] Pass if no findings from cellate (for now)
- [ ] View res

In [ ]:
seen = []
for article in data_json:
    for annotation in article["annotations"]:
        text = annotation["exact"]
        if text not in seen:
            seen.append(text)
            print(text)

Within our dataset, we did not observe PD1-PDL1 interactions in either of the tumour subtypes (Fig. 2C, D).
Considering that some of the STAB1 signature genes are typically expressed by foetal Mɸ (such as STAB1, FOLR2, SLC40A1, MERTK, GPR34 and F13A1)54, we wanted to explore if further transcriptional commonalities exist between tumour-originating STAB1 + Mɸ and Mɸ isolated from human foetal lung.
STAB1 + Mɸ displayed a transcriptional resemblance to CAMLs, which concurrently expressed genes associated with both Mɸ and epithelial cells, and exhibited copy number alterations (CNAs) similar to those found in tumour cells.
Our initial analysis suggests that other ICIs (such as CTLA4, TIGIT, LILRB1/2 and TIM3) might be promising targets in the treatment of NSCLC.
Ferroportin-mediated release of free iron by M2 Mɸ was reported to promote the proliferation of renal carcinoma cells in vitro, possibly by supporting the high iron requirement due to increased DNA synthesis52.
To assess the level